<a href="https://colab.research.google.com/github/EhsanGhasemi423/INFS-8368/blob/main/6_Neural_Networks_Part_2_(Regression_Synthetic_Data_Loading_Minibatches).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Synthetic Regression Data

Real-world datasets are often large, complex, and may contain unknown relationships between the input features and the target values. Before applying machine learning models to real data, it is useful to test them on **synthetic data**.

A **synthetic dataset** is an artificially generated dataset created using a known mathematical model. Because the true model parameters are known, synthetic data provides a simple way to verify that a learning algorithm and its implementation are working correctly.


In [ ]:
!pip install -q d2l --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 1.7 MB/s eta 0:00:00


In [ ]:
%matplotlib inline
import random
import tensorflow as tf
from d2l import tensorflow as d2l


## Generating the Dataset

To demonstrate linear regression, we generate a **synthetic dataset** containing **1,000 training examples**, each with **two input features** drawn from a standard normal distribution.

The target value for each example is computed using a linear model with added random noise:

$$
y=\mathbf{Xw}+b+\varepsilon,
$$

where:

- $\mathbf{X}$ is the feature matrix,
- $\mathbf{w}$ is the vector of true weights,
- $b$ is the true bias, and
- $\varepsilon$ is random noise.

The noise is sampled from a normal distribution with mean $\mu=0$ and standard deviation $\sigma=0.01$. Adding a small amount of noise makes the data more realistic, since real-world observations rarely follow an exact linear relationship.

Because the true weights and bias are known, we can later compare them with the values learned by the model to verify that our implementation works correctly.

In [ ]:
class SyntheticRegressionData(d2l.DataModule):
    """Synthetic data for linear regression."""
    def __init__(self, w, b, noise=0.01, num_train=1000, num_val=1000,
                 batch_size=32):
        super().__init__()
        self.save_hyperparameters()
        n = num_train + num_val
        self.X = tf.random.normal((n, w.shape[0]))
        noise = tf.random.normal((n, 1)) * noise
        self.y = tf.matmul(self.X, tf.reshape(w, (-1, 1))) + b + noise

In [ ]:
data = SyntheticRegressionData(w=tf.constant([2, -3.4]), b=4.2)

In [ ]:
print('features:', data.X[0],'\nlabel:', data.y[0])

features: tf.Tensor([ 0.36519122 -0.8925901 ], shape=(2,), dtype=float32) 
label: tf.Tensor([7.9708962], shape=(1,), dtype=float32)


In [ ]:
print("First example")
print("Features:", data.X[0].numpy())
print("Label:", data.y[0].numpy())

First example
Features: [ 0.36519122 -0.8925901 ]
Label: [7.9708962]


In [ ]:
data.X.shape, data.y.shape

(TensorShape([2000, 2]), TensorShape([2000, 1]))

## Loading Data in Mini-Batches

During training, we do not process the entire dataset at once. Instead, the training examples are divided into smaller groups called **mini-batches**.

A mini-batch contains a subset of the training examples and their corresponding labels. Processing data in mini-batches makes training more efficient and is the standard approach used in deep learning.

TensorFlow provides the `tf.data.Dataset` API to create and manage mini-batches.

In [ ]:
import tensorflow as tf
#The dataset currently contains all training examples. During training, TensorFlow groups these examples into mini-batches so that the model processes a small subset of the data at each iteration.
train_dataset = tf.data.Dataset.from_tensor_slices(
    (data.X[:data.num_train], data.y[:data.num_train])
)

train_dataset = train_dataset.shuffle(
    buffer_size=data.num_train
).batch(data.batch_size)

In [ ]:
X, y = next(iter(train_dataset))

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (32, 2)
y shape: (32, 1)


> **Note:** The D2L library provides abstractions that organize data generation and loading into reusable classes. However, these abstractions are not required. We can generate the same synthetic regression dataset directly using TensorFlow.

In [ ]:
import tensorflow as tf

# True model parameters
true_w = tf.constant([2.0, -3.4])
true_b = 4.2

# Generate 1000 examples with 2 features
X = tf.random.normal((1000, 2))

# Add small random noise
noise = tf.random.normal((1000, 1), mean=0.0, stddev=0.01)

# Generate labels using y = Xw + b + noise
y = (
    tf.matmul(X, tf.reshape(true_w, (-1, 1)))
    + true_b
    + noise
)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFirst example:")
print("Features:", X[0].numpy())
print("Label:", y[0].numpy())

X shape: (1000, 2)
y shape: (1000, 1)

First example:
Features: [-1.5122005 -0.6240861]
Label: [3.2926896]


In [ ]:
batch_size = 32

train_dataset = (
    tf.data.Dataset
    .from_tensor_slices((X, y))
    .shuffle(buffer_size=len(X))
    .batch(batch_size)
)

In [ ]:
X_batch, y_batch = next(iter(train_dataset))

print("Mini-batch feature shape:", X_batch.shape)
print("Mini-batch label shape:", y_batch.shape)

print("\nFirst example in the mini-batch")
print("Features:", X_batch[0].numpy())
print("Label:", y_batch[0].numpy())

Mini-batch feature shape: (32, 2)
Mini-batch label shape: (32, 1)

First example in the mini-batch
Features: [-0.37918082 -0.0120737 ]
Label: [3.476141]


In [ ]:
print("\nFirst example in the mini-batch")
print("Features:", X_batch[0].numpy())
print("Label:", y_batch[0].numpy())


First example in the mini-batch
Features: [-0.37918082 -0.0120737 ]
Label: [3.476141]


In [ ]:
batch = next(iter(train_dataset))

X_batch = batch[0]
y_batch = batch[1]
print("Mini-batch feature shape:", X_batch.shape)
print("Mini-batch label shape:", y_batch.shape)

print("\nFirst example in the mini-batch")
print("Features:", X_batch[0].numpy())
print("Label:", y_batch[0].numpy())

Mini-batch feature shape: (32, 2)
Mini-batch label shape: (32, 1)

First example in the mini-batch
Features: [-0.34226006 -0.9516356 ]
Label: [6.7443385]


Both approaches produce the same type of data. The direct TensorFlow approach is more transparent, while the D2L abstraction can make larger projects easier to organize.

## Summary

In this section, we created a **synthetic dataset** for a linear regression problem.

- We defined the **true weights** and **bias** that describe the underlying linear relationship.
- We generated **1,000 training examples**, each with **two input features**, using random values drawn from a normal distribution.
- We computed the corresponding **target values (labels)** using the linear regression model

  $$
  y = Xw + b + \text{noise},
  $$

  where a small amount of random noise was added to simulate real-world data.

The generated data consisted of two tensors:

- `X`: the feature matrix with shape `(1000, 2)`.
- `y`: the label vector with shape `(1000, 1)`.

Next, we converted these tensors into a **TensorFlow Dataset**:

```python
train_dataset = tf.data.Dataset.from_tensor_slices((X, y))
```

A **TensorFlow Dataset** is **not another tensor**. Instead, it is a TensorFlow object that organizes the data by pairing each feature vector with its corresponding label:

```text
(X[0], y[0])
(X[1], y[1])
(X[2], y[2])
...
```

This makes it easy for TensorFlow to efficiently prepare the data for training.

We then prepared the dataset by:

- **Shuffling** the training examples so they are presented in a random order.
- **Grouping** the examples into **mini-batches** of 32 examples:

```python
train_dataset = train_dataset.shuffle(len(X)).batch(32)
```

During training, the model processes **one mini-batch at a time** instead of the entire dataset. Each mini-batch contains two tensors:

- `features`: a tensor containing the input features with shape `(32, 2)`.
- `labels`: a tensor containing the corresponding target values with shape `(32, 1)`.

For example:

```python
features, labels = next(iter(train_dataset))
```

The variables `features` and `labels` represent **one mini-batch** of data. The first example in the mini-batch can be accessed as:

```python
print(features[0].numpy())
print(labels[0].numpy())
```

Using mini-batches makes training more computationally efficient and is the standard approach in modern machine learning and deep learning.